# Oban yield pilot: evaluation receipt

The one-month gradient-boosting model improved held-out MAE by 9.7%. Two- and three-month models underperformed the seasonal baseline. Keep those baselines. This is a historical pilot, with production only through July 2026. No current scheduling or fertiliser-dose recommendation is justified.

Source data stays local. The trained model and 12,939 held-out predictions are retained alongside this notebook.

## Method and limitations

Monthly source-block FFB tonnes are the target. Training through 2023; selection and empirical error-band calibration in 2024; held-out testing in 2025–July 2026. Horizons use separate models and an origin embargo. Features include known production/rainfall lags and fertiliser totals lagged two calendar years. Publication dates are unavailable: month-end availability is an assumption. Current source-to-map crosswalks may omit historical changes. See scripts/model/README.md and train_yield.py for full reproducibility.

MAE is tonnes per source-month, not percentage accuracy. Cross-block observations share weather and are not independent climate samples. Reported ranges are empirical, and their measured coverage is below the nominal 80%.

In [1]:
from pathlib import Path
import json, csv
root = Path.cwd()
while not (root / 'scripts/model/train_yield.py').exists():
    if root.parent == root: raise RuntimeError('Run inside the digitalpalm-mapping project')
    root = root.parent
report = json.loads(next((root / 'data/yield-models').glob('*.json')).read_text())
print(json.dumps(report['audit'], indent=2))

{
  "inputMonthlyRows": 22788,
  "usableMonthlyRows": 19621,
  "latestObservedMonth": "2026-07",
  "latestSources": 247,
  "forecastSources": 246,
  "excludedDuplicateRows": 0,
  "rainfallObservedMonths": 112
}


In [2]:
print('Horizon | Model MAE | Baseline MAE | Error reduction | Test range coverage')
for h in report['horizons']:
    print(f"{h['monthsAhead']} months | {h['modelMetrics']['maeTonnes']:.2f} t | {h['baselineMetrics']['maeTonnes']:.2f} t | {h['improvementPercent']:.1f}% | {h['testRangeCoveragePercent']:.1f}%")

Horizon | Model MAE | Baseline MAE | Error reduction | Test range coverage
1 months | 7.30 t | 8.08 t | 9.7% | 77.5%
2 months | 8.52 t | 7.93 t | -7.4% | 75.3%
3 months | 9.22 t | 7.87 t | -17.2% | 72.8%


In [3]:
rows = list(csv.DictReader((root / 'data/yield-models/heldout-predictions.csv').open()))
for h in report['horizons']:
    selected = [r for r in rows if int(r['horizon']) == h['monthsAhead']]
    mae = sum(abs(float(r['actual'])-float(r['prediction'])) for r in selected)/len(selected)
    assert abs(mae-h['modelMetrics']['maeTonnes']) < 1e-9
print(f'Independently reconciled {len(rows):,} held-out predictions with the saved model metrics.')

Independently reconciled 12,939 held-out predictions with the saved model metrics.


## Recommended next steps

1. Import August and September actual production and retrain.
2. Retain the seasonal baseline for horizons that fail the 5% improvement gate.
3. Use the one-month model only as a field-reviewed pilot; its held-out weighted absolute percentage error is about 40.2%, so it is not precise enough to justify automatic operational changes.
4. Confirm effective hectares for sources without them.
5. Collect dated nutrient applications and soil/leaf observations before considering fertiliser response scenarios.

The app keeps source predictions separate, reports coverage, and suppresses operational crop alerts when the model snapshot is stale.